In [1]:
import os
import numpy as np
import pandas as pd
import utils as utils #커스텀 패키지
import torch
import torch.nn as nn
from transformers import AutoTokenizer

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
config = utils.load_config("./config.yaml")

#이전 데이터 셋 사용
train_dataset = pd.read_csv(config["data"]["path"] + config["data"]["train_path"])

#측면에 대한 정답 레이블 생성
train_dataset['AspectConfidence'] = 1
train_dataset.loc[:,'SentimentPolarity'] += 1

#Aspect가 소비전력인 데이터 삭제
train_dataset = train_dataset[train_dataset['Aspect'] != '소비전력']

#8:2로 train/valid 분리
train_data = train_dataset.sample(frac=0.8, random_state=config['seed'])
valid_data = train_dataset.drop(train_data.index)

c:\Users\user\.conda\envs\review_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 전처리

In [2]:
#측면만 따로 뽑아서 확인
ASPECTS = train_data['Aspect'].unique()
aspect2id = {
    aspect: idx
    for idx, aspect in enumerate(ASPECTS)
}
print(ASPECTS)

#측면을 숫자로 변환
valid_data.loc[:, 'Aspect'] = (valid_data['Aspect'].map(aspect2id))

['화질' '음량/음질' '무게' '품질' '시간/속도' '편의성' '디자인' '가격' '사이즈' '기능' '제조일/제조사'
 '제품구성' '배터리' '조작성' '내구성' '색상' '소음' '소재' '용량']


In [3]:
from dataset import create_negative_aspect_samples

#부정 셋 생성
negative_data = create_negative_aspect_samples(train_data, config['seed'])

#학습 셋과 합치기
train_data = pd.concat([train_data, pd.DataFrame(negative_data)], ignore_index=True)
train_data.describe()

,SentimentPolarity,가격,기능,내구성,디자인,무게,배터리,사이즈,색상,소비전력,...,시간/속도,용량,음량/음질,제조일/제조사,제품구성,조작성,편의성,품질,화질,AspectConfidence
count,124932.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.0,...,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,62466.000000,124932.000000
mean,0.261686,0.138940,0.120177,0.015913,0.059328,0.038917,0.019915,0.069734,0.050267,0.0,...,0.035235,0.011670,0.058624,0.035940,0.043992,0.075289,0.074809,0.069046,0.055934,0.500000
std,1.392562,0.345886,0.325171,0.125139,0.236240,0.193399,0.139709,0.254700,0.218498,0.0,...,0.184375,0.107398,0.234921,0.186141,0.205079,0.263859,0.263085,0.253534,0.229797,0.500002
min,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
75%,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
aspect_sentiment_ratio = pd.crosstab(
    train_data['Aspect'],
    train_data['SentimentPolarity'],
    normalize='index'
)

print(aspect_sentiment_ratio)

SentimentPolarity   -1         0         1         2
Aspect                                              
가격                 0.5  0.046780  0.011407  0.441814
기능                 0.5  0.068203  0.014120  0.417677
내구성                0.5  0.299799  0.023642  0.176559
디자인                0.5  0.063681  0.010523  0.425796
무게                 0.5  0.172974  0.025093  0.301933
배터리                0.5  0.243167  0.012460  0.244373
사이즈                0.5  0.103994  0.017103  0.378903
색상                 0.5  0.097930  0.009554  0.392516
소음                 0.5  0.192793  0.022973  0.284234
소재                 0.5  0.307910  0.016008  0.176083
시간/속도              0.5  0.126533  0.018401  0.355066
용량                 0.5  0.182442  0.028807  0.288752
음량/음질              0.5  0.104451  0.031404  0.364145
제조일/제조사            0.5  0.087082  0.005345  0.407572
제품구성               0.5  0.183588  0.016012  0.300400
조작성                0.5  0.140761  0.019349  0.339889
편의성                0.5  0.111385  0.009630  0.

In [5]:
#자기 모델에 맞게 수정
tokenizer = AutoTokenizer.from_pretrained(
    "monologg/koelectra-base-v3-discriminator"
)

In [6]:
from torch.utils.data import DataLoader
from dataset import ABSADataset

#모델 학습용 데이터셋으로 변환
train_data_set = ABSADataset(train_data, tokenizer)

train_loader = DataLoader(
    train_data_set,
    batch_size=config["training"]["batch_size"],
    shuffle=True
)

#### 학습 및 검정

In [7]:
from sklearn.utils.class_weight import compute_class_weight

#감정에 대한 불균형 데이터를 위해 가중치 조절 계산
sentiment = train_data[train_data['SentimentPolarity'] != -1]['SentimentPolarity']

sentiment_weight = compute_class_weight(class_weight='balanced', 
                                            classes=np.unique(sentiment), y=sentiment)

sentiment_weight = torch.tensor(sentiment_weight, dtype=torch.float32).to(device)

print(sentiment_weight)

#Loss 함수 생성
criterion_aspect = nn.CrossEntropyLoss()

#가중치 적용
criterion_sentiment = nn.CrossEntropyLoss(ignore_index=-1, weight=sentiment_weight)

tensor([ 1.4972, 10.6289,  0.4468], device='cuda:0')


In [8]:
from sklearn.metrics import f1_score
from utils import evaluation_dataset
from model import ABSAModel, training_model

#모델 생성
model = ABSAModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["optimizer"]["lr"]
)

epochs = config["training"]["epoch"]

#학습 재개 시
if config['train_resume'] == True:
    checkpoint = torch.load(
        config['training']["checkpoint"]["path"],
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    epoch = checkpoint["epoch"]
    best_epoch = checkpoint["best_epoch"]
    best_f1_score = checkpoint["best_f1_score"]
    
else:
    epoch = 0
    best_epoch = 0
    best_f1_score = 0

for epoch in range(epoch, epochs):
    print(f"epoch {epoch + 1}/{epochs}")
    
    avg_loss, aspect_acc, sentiment_acc = training_model(model, optimizer, train_loader, 
                                                         criterion_aspect, criterion_sentiment, device)
    
    #데이터 학습 결과 출력
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")
    print(f"Train Aspect Acc : {aspect_acc:.4f}")
    print(f"Train Sentiment Acc : {sentiment_acc:.4f}")
    
    #검증 데이터로 과적합 및 성능 확인
    all_aspect_preds, all_aspect_labels, all_sentiment_preds, all_sentiment_labels = evaluation_dataset(model, tokenizer, valid_data, ASPECTS, device)
    aspect_f1 = f1_score(all_aspect_labels, all_aspect_preds, average='weighted')
    sentiment_f1 = f1_score(all_sentiment_labels, all_sentiment_preds, average='macro')
    
    print(f"Valid Aspect    F1 Score(weighted): {aspect_f1}")
    print(f"Valid Sentiment F1 Score(macro): {sentiment_f1}")
    
    cur_f1_score = (aspect_f1 * 0.4) + (sentiment_f1 * 0.6)
    
    #sentiment f1 score 기준으로 잘 나온 모델을 저장
    if cur_f1_score > best_f1_score:
        best_f1_score = cur_f1_score
        best_epoch = epoch
        print("Best_Model_Changed")
        torch.save({
            "model_state_dict": model.state_dict(),
        }, config["model"]["save_path"])

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch + 1,
        "best_epoch": best_epoch,
        "best_f1_score": best_f1_score
    }, config['training']["checkpoint"]["path"])

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 38654.47it/s]
[transformers] ElectraModel LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/5


100%|██████████| 3905/3905 [12:15<00:00,  5.31it/s, loss=0.4315, aspect=0.0080, sentiment=0.3276, a_acc=0.9541, s_acc=0.9495]



Epoch 1 Average Loss: 0.0134
Train Aspect Acc : 0.9541
Train Sentiment Acc : 0.9495


100%|██████████| 15617/15617 [03:57<00:00, 65.64it/s]


...

Valid Aspect    F1 Score(weighted): 0.9055267308027266
Valid Sentiment F1 Score(macro): 0.8721550498939162
Best_Model_Changed
epoch 5/5


100%|██████████| 3905/3905 [12:16<00:00,  5.30it/s, loss=0.0090, aspect=0.0076, sentiment=0.0029, a_acc=0.9886, s_acc=0.9852]



Epoch 5 Average Loss: 0.0035
Train Aspect Acc : 0.9886
Train Sentiment Acc : 0.9852


100%|██████████| 15617/15617 [03:54<00:00, 66.51it/s]


Valid Aspect    F1 Score(weighted): 0.9076016933695373
Valid Sentiment F1 Score(macro): 0.8491268602858342


In [9]:
#best 모델 불러오기
save_model = torch.load(
    config['model']["save_path"],
    map_location=device
)

model.load_state_dict(
    save_model["model_state_dict"]
)

<All keys matched successfully>

In [11]:
from utils import print_evaluation_report

#Valid 데이터 평가
model.eval()

all_aspect_preds, all_aspect_labels, all_sentiment_preds, all_sentiment_labels = evaluation_dataset(model, tokenizer, valid_data, ASPECTS, device)
print_evaluation_report(all_aspect_preds, all_aspect_labels, all_sentiment_preds, all_sentiment_labels, ASPECTS)

100%|██████████| 15617/15617 [03:39<00:00, 71.12it/s]

              precision    recall  f1-score   support

    negative       0.96      0.96      0.96      3424
     neutral       0.60      0.75      0.66       499
    positive       0.99      0.98      0.99     11694

    accuracy                           0.97     15617
   macro avg       0.85      0.90      0.87     15617
weighted avg       0.98      0.97      0.97     15617

              precision    recall  f1-score   support

          화질       0.94      0.95      0.95       851
       음량/음질       0.94      0.92      0.93       936
          무게       0.94      0.99      0.97       598
          품질       0.89      0.86      0.88      1069
       시간/속도       0.88      0.93      0.91       548
         편의성       0.84      0.76      0.80      1208
         디자인       0.95      0.97      0.96       900
          가격       0.98      0.97      0.98      2173
         사이즈       0.94      0.97      0.95      1083
          기능       0.88      0.82      0.85      1861
     제조일/제조사       0.94 

#### 학습 + 검증 데이터 학습, Test 셋 성능 확인

In [ ]:
#train_valid 데이터 생성
train_valid_dataset = pd.concat([train_data, valid_data], ignore_index=True)

train_valid_ds = ABSADataset(train_valid_dataset, tokenizer, ASPECTS)

train_valid_loader = DataLoader(
    train_valid_ds,
    batch_size=config["training"]["batch_size"],
    shuffle=False
)

In [ ]:
#모델 생성
model = ABSAModel().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["optimizer"]["lr"]
)

epochs = config["training"]["epoch"]

#학습 재개 시
if config['train_resume'] == True:
    checkpoint = torch.load(
        config['training']["checkpoint"]["path"],
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    epoch = checkpoint["epoch"]
    best_epoch = checkpoint["best_epoch"]
    
else:
    epoch = 0
    best_epoch = 0

for epoch in range(epoch, best_epoch):
    print(f"epoch {epoch + 1}/{best_epoch}")
    
    avg_loss, aspect_acc, sentiment_acc = training_model(model, optimizer, train_valid_loader, 
                                                         criterion_aspect, criterion_sentiment, device)
    
    #데이터 학습 결과 출력
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")
    print(f"Train Aspect Acc : {aspect_acc:.4f}")
    print(f"Train Sentiment Acc : {sentiment_acc:.4f}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch + 1,
        "best_epoch": best_epoch,
    }, config['training']["checkpoint"]["path"])
    
print("Model_Saved")
torch.save({
    "model_state_dict": model.state_dict(),
}, config["model"]["train_valid_path"])

In [ ]:
# test 셋 준비(모델 선택 완료 후 사용)
# test_dataset = pd.read_csv(config['data']['path'] + config['data']['val_path'])

# #Aspect가 소비전력인 데이터 삭제
# test_dataset = test_dataset[test_dataset['Aspect'] != '소비전력']
# test_dataset.drop(['소비전력'], axis=1, inplace=True)

# #측면에 대한 정답 레이블 생성
# test_dataset.loc[:,'AspectConfidence'] = 1
# test_dataset.loc[:, "SentimentPolarity"] += 1
 
# test_data = test_dataset[['SentimentText', 'Aspect', 'SentimentPolarity', 'AspectConfidence']]

# #Aspect를 aspects 기준으로 index 변환
# test_data.loc[:, 'Aspect'] = (test_data['Aspect'].map(aspect2id))

# #Test 데이터 평가
# model.eval()
# all_aspect_preds, all_aspect_labels, all_sentiment_preds, all_sentiment_labels = evaluation_dataset(model, tokenizer, test_data, ASPECTS, device)
# print_evaluation_report(all_aspect_preds, all_aspect_labels, all_sentiment_preds, all_sentiment_labels)

#### 예시 리뷰 추론 확인

In [ ]:
model.eval()
sentence = '가격은 싼데 품질이 아쉽네요'

THRESHOLD = 0.8
USE_FALLBACK_TOP1 = False  # True면 아무 것도 없을 때 최고점 1개 강제 선택

with torch.no_grad():
    texts = [sentence] * len(ASPECTS)
    aspect_texts = list(ASPECTS)

    encoding = tokenizer(
        texts,
        aspect_texts,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    aspect_logits, sentiment_logits = model(input_ids, attention_mask)

    aspect_probs = torch.softmax(aspect_logits, dim=-1)[:, 1].cpu().numpy()
    sentiment_pred = torch.argmax(sentiment_logits, dim=-1).cpu().numpy()

    selected_idx = np.where(aspect_probs >= THRESHOLD)[0]

    if len(selected_idx) == 0 and USE_FALLBACK_TOP1:
        selected_idx = np.array([int(np.argmax(aspect_probs))])

    final_aspect_pred = ASPECTS[selected_idx]
    final_sentiment_pred = [
        ["negative", "neutral", "positive"][int(i)]
        for i in sentiment_pred[selected_idx]
    ]
    final_aspect_prob = [float(aspect_probs[i]) for i in selected_idx]

    print(final_aspect_pred)
    print(final_sentiment_pred)
    print(final_aspect_prob)